<a href="https://colab.research.google.com/github/pcmouadji-dot/deep_learning/blob/main/comment_toxic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Bidirectional, Dense, Embedding

In [4]:
df=pd.read_csv('train.csv', engine='python', on_bad_lines='skip')
df.head()
#df[df['toxic']==1].head()


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
x=df['comment_text']
y=df[df.columns[2:]].values



In [6]:
x.values.shape

(2429,)

In [7]:
vecto=TextVectorization(max_tokens=250000,output_mode='int', output_sequence_length=1800)
vecto.adapt(x.values)


In [8]:
vecto('bitch')

<tf.Tensor: shape=(1800,), dtype=int64, numpy=array([1800,    0,    0, ...,    0,    0,    0])>

In [9]:
vec_text=vecto(x.values)
vec_text

<tf.Tensor: shape=(2429, 1800), dtype=int64, numpy=
array([[  679,    85,     2, ...,     0,     0,     0],
       [15131,    61,  1559, ...,     0,     0,     0],
       [  512,   385,    66, ...,     0,     0,     0],
       ...,
       [   85,    45,   116, ...,     0,     0,     0],
       [    8,    73,  7053, ...,     0,     0,     0],
       [    2,   599,     9, ...,     0,     0,     0]])>

In [10]:
dataset=tf.data.Dataset.from_tensor_slices((vec_text,y))
dataset=dataset.cache()
dataset=dataset.shuffle(90000)
dataset=dataset.batch(9)
dataset=dataset.prefetch(5)

In [11]:
train=dataset.take(int(len(dataset)*.7))
val=dataset.skip(int(len(dataset)*.7)).take(int(len(dataset)*.2))
test=dataset.skip(int(len(dataset)*.9)).take(int(len(dataset)*.1))

In [12]:
train_generater=train.as_numpy_iterator()
train_generater.next()

(array([[  482,  9991,    43, ...,     0,     0,     0],
        [   14,   217,    23, ...,     0,     0,     0],
        [    8,   233,    22, ...,     0,     0,     0],
        ...,
        [13908, 14255,     8, ...,     0,     0,     0],
        [ 2257,   861,  2258, ...,     0,     0,     0],
        [ 3202,     4,  2624, ...,     0,     0,     0]]),
 array([[1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]]))

In [13]:
model=Sequential([
    Embedding(250000+1,32),
    Bidirectional(LSTM(32,activation='tanh')),
    Dense(128,activation='relu'),
    Dense(256,activation='relu'),
    Dense(128,activation='relu'),
    Dense(6,activation='sigmoid')


])
model.compile(loss='BinaryCrossentropy',optimizer='Adam')

model.summary()#before trainning

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
history=model.fit(train,epochs=7,validation_data=val)
model.summary()

Epoch 1/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 28s 118ms/step - loss: 0.1822 - val_loss: 0.1397
Epoch 2/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 22s 117ms/step - loss: 0.0975 - val_loss: 0.0627
Epoch 3/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 21s 112ms/step - loss: 0.0697 - val_loss: 0.0450
Epoch 4/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 22s 115ms/step - loss: 0.0572 - val_loss: 0.0393
Epoch 5/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 22s 115ms/step - loss: 0.0456 - val_loss: 0.0345
Epoch 6/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 21s 111ms/step - loss: 0.0328 - val_loss: 0.0354
Epoch 7/7
189/189 ━━━━━━━━━━━━━━━━━━━━ 22s 114ms/step - loss: 0.0386 - val_loss: 0.0282


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 1800, 32)       │     8,000,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,275,060 (92.60 MB)

 Trainable params: 8,091,686 (30.87 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 16,183,374 (61.73 MB)

In [17]:
txt=vecto('u are gay')
res=model.predict(np.expand_dims(txt,0))
res

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step


array([[0.64051616, 0.00859695, 0.10287508, 0.01304626, 0.07752403,
        0.01683611]], dtype=float32)

In [18]:
from tensorflow.keras.metrics import Precision,Recall,CategoricalAccuracy
pre=Precision()
re=Recall()
acc=CategoricalAccuracy()


In [19]:
for batch in test.as_numpy_iterator():
  x_true,y_true=batch
  yhat=model.predict(x_true)
  y_true=y_true.flatten()
  yhat=yhat.flatten()
  pre.update_state(y_true,yhat)
  re.update_state(y_true,yhat)
  acc.update_state(y_true,yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━

In [20]:
print(pre.result(),re.result(),acc.result())

tf.Tensor(0.9259259, shape=(), dtype=float32) tf.Tensor(0.6849315, shape=(), dtype=float32) tf.Tensor(0.4814815, shape=(), dtype=float32)
